Сравниваю
`TopPopular` и  `ClusterTopPopular` - тот же TopPopular, но с большей вариативностью по кластеризации аудиофич, чтобы выйти на более полезные первые интераакции так как мы не можем сделать опросник по жанрам


In [18]:
import json
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from datasets import load_dataset, load_from_disk
from sklearn.cluster import KMeans

In [2]:
EVENTS_PATH = "yambda_events"
EMBEDDINGS_PATH = "yambda_embedings"

OUTPUT_DIR = Path("model_cold_start")

NUM_CLUSTERS = 30
TOP_PER_CLUSTER = 1000
RANDOM_STATE = 42

TEST_FRAC_BY_TIME = 0.2
LISTEN_MIN = 0.5

K_LIST = [10, 20, 50, 100, 500]

LOG_MLFLOW = False
EXPERIMENT_NAME = "cold_start"
RUN_NAME = "ClusterTopPopular"

EVENTS = {"listen", "like", "dislike", "unlike", "undislike"}
POSITIVE_EVENTS = {"listen", "like"}

In [3]:
if not Path(EVENTS_PATH).exists():
    events_ds = load_dataset(
        "yandex/yambda",
        data_dir="flat/50m",
        data_files="multi_event.parquet",
    )
    events_ds.save_to_disk(EVENTS_PATH)

if not Path(EMBEDDINGS_PATH).exists():
    emb_ds = load_dataset(
        "yandex/yambda",
        data_files="embeddings.parquet",
    )
    emb_ds.save_to_disk(EMBEDDINGS_PATH)

events_ds = load_from_disk(EVENTS_PATH)
emb_ds = load_from_disk(EMBEDDINGS_PATH)

events = events_ds["train"].to_pandas()
emb = emb_ds["train"].to_pandas()

events.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,100,39420,8326270,0,100.0,170.0,listen
1,100,39420,1441281,0,100.0,105.0,listen
2,100,39625,286361,0,100.0,185.0,listen
3,100,40110,732449,0,100.0,240.0,listen
4,100,40360,3397170,0,46.0,130.0,listen


In [4]:
emb.head()

,item_id,embed,normalized_embed
0,2,"[-1.5340346097946167, -0.3667665123939514, 3.2...","[-0.06463761062776703, -0.015453960991530893, ..."
1,3,"[-3.7614669799804688, -1.0682544708251953, 3.3...","[-0.16393688999107472, -0.04655798298860864, 0..."
2,4,"[2.445533037185669, -2.5236034393310547, 5.115...","[0.07627247358910043, -0.07870737125565455, 0...."
3,5,"[0.8328456878662109, 0.11612524837255478, -0.6...","[0.03149010106338991, 0.004390724308883539, -0..."
4,6,"[-2.431483030319214, -0.5687204003334045, -0.2...","[-0.10345024525108451, -0.024196864284124445, ..."


In [5]:
events = events[events["event_type"].isin(EVENTS)].copy()

events["played_ratio"] = events["played_ratio_pct"].astype(float) / 100
events["played_ratio"] = events["played_ratio"].clip(upper=1.0)

listen_mask = (events["event_type"] == "listen") & (events["played_ratio"] >= LISTEN_MIN)
events = events[(events["event_type"] != "listen") | listen_mask].copy()

ITEM_COL = "item_id"
EMB_COL = "normalized_embed"

item_ids = emb[ITEM_COL].astype(int).to_numpy()
item_embeddings = np.vstack(emb[EMB_COL].to_numpy()).astype("float32")

events = events[events["item_id"].isin(set(item_ids))].copy()
events = events.sort_values("timestamp").reset_index(drop=True)

events.shape, item_embeddings.shape

((29761990, 8), (7721749, 128))

In [6]:
split_ts = events["timestamp"].quantile(1 - TEST_FRAC_BY_TIME)

train_events = events[events["timestamp"] <= split_ts].copy()
test_events = events[events["timestamp"] > split_ts].copy()

test_positive = test_events[test_events["event_type"].isin(POSITIVE_EVENTS)].copy()

train_events.shape, test_events.shape, test_positive.shape

((23809595, 8), (5952395, 8), (5888419, 8))

# TopPopular

In [7]:
listen_count = (
    train_events[train_events["event_type"] == "listen"]
    .groupby("item_id")
    .size()
)

liked = set()
disliked = set()

for row in train_events.itertuples(index=False):
    key = (int(row.uid), int(row.item_id))

    if row.event_type == "like":
        liked.add(key)
        disliked.discard(key)

    elif row.event_type == "unlike":
        liked.discard(key)

    elif row.event_type == "dislike":
        disliked.add(key)
        liked.discard(key)

    elif row.event_type == "undislike":
        disliked.discard(key)

In [8]:
like_count = defaultdict(int)
dislike_count = defaultdict(int)

for _, item_id in liked:
    like_count[item_id] += 1

for _, item_id in disliked:
    dislike_count[item_id] += 1

all_items = set(listen_count.index) | set(like_count) | set(dislike_count)

item_popularity = {}

for item_id in all_items:
    item_popularity[int(item_id)] = float(
        np.log1p(listen_count.get(item_id, 0))
        + 3 * np.log1p(like_count.get(item_id, 0))
        - 3 * np.log1p(dislike_count.get(item_id, 0))
    )

top_popular = [
    int(item_id)
    for item_id, _ in sorted(item_popularity.items(), key=lambda x: x[1], reverse=True)
]

pd.Series(item_popularity).sort_values(ascending=False).head(20)

602720     19.658752
3762840    18.886696
1360474    18.763521
9342289    18.736452
2818520    18.597619
8312981    18.428008
3899874    18.372817
2784011    18.320024
1638622    18.303086
8323008    18.291201
3911887    18.248700
5589983    18.191223
7041449    18.178342
7182524    18.141950
7610514    18.136783
1404151    18.104428
3375935    18.093777
5315077    18.076267
8621338    18.063487
5436843    18.048623
dtype: float64

# KMeans

In [9]:
kmeans = KMeans(
    n_clusters=NUM_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init="auto",
)

clusters = kmeans.fit_predict(item_embeddings)

item_to_cluster = {
    int(item_id): int(cluster)
    for item_id, cluster in zip(item_ids, clusters)
}

cluster_sizes = pd.Series(clusters).value_counts().sort_index()
cluster_sizes.describe()

count        30.000000
mean     257391.633333
std       61556.978283
min      139645.000000
25%      218495.250000
50%      251462.000000
75%      309366.750000
max      379409.000000
Name: count, dtype: float64

In [10]:
cluster_items = defaultdict(list)

for item_id, cluster in zip(item_ids, clusters):
    item_id = int(item_id)
    cluster = int(cluster)
    score = item_popularity.get(item_id, 0.0)
    cluster_items[cluster].append((item_id, score))

cluster_top_items = {}

for cluster, items in cluster_items.items():
    items = sorted(items, key=lambda x: x[1], reverse=True)
    cluster_top_items[cluster] = [item_id for item_id, _ in items[:TOP_PER_CLUSTER]]

cluster_score = {}

for cluster, items in cluster_top_items.items():
    scores = [item_popularity.get(item_id, 0.0) for item_id in items[:20]]
    cluster_score[cluster] = float(np.mean(scores))

ordered_clusters = [
    cluster
    for cluster, _ in sorted(cluster_score.items(), key=lambda x: x[1], reverse=True)
]

ordered_clusters[:10]

[0, 29, 12, 26, 23, 20, 2, 3, 22, 27]

In [11]:
def recommend_top_popular(k=20, seen_items=None):
    seen_items = seen_items or set()
    return [x for x in top_popular if x not in seen_items][:k]


def recommend_cluster_div(k=20, seen_items=None):
    seen_items = seen_items or set()

    recs = []
    used = set()
    ptr = {cluster: 0 for cluster in ordered_clusters}

    while len(recs) < k:
        added = False

        for cluster in ordered_clusters:
            items = cluster_top_items[cluster]
            i = ptr[cluster]

            while i < len(items) and (items[i] in seen_items or items[i] in used):
                i += 1

            ptr[cluster] = i

            if i >= len(items):
                continue

            item_id = items[i]

            recs.append(item_id)
            used.add(item_id)
            ptr[cluster] += 1
            added = True

            if len(recs) == k:
                break

        if not added:
            break

    return recs

In [12]:
recs = recommend_cluster_div(k=20)

pd.DataFrame({
    "rank": range(1, len(recs) + 1),
    "item_id": recs,
    "cluster": [item_to_cluster[x] for x in recs],
    "popularity": [item_popularity.get(x, 0.0) for x in recs],
})

,rank,item_id,cluster,popularity
0,1,1360474,0,18.763521
1,2,7041449,29,18.178342
2,3,5436843,12,18.048623
3,4,2818520,26,18.597619
4,5,602720,23,19.658752
5,6,8312981,20,18.428008
6,7,7182524,2,18.141950
7,8,8323008,3,18.291201
8,9,3206857,22,17.892334
9,10,3911887,27,18.248700


In [13]:
test_user_items = (
    test_positive
    .groupby("uid")["item_id"]
    .apply(lambda x: set(map(int, x)))
)

train_user_seen = (
    train_events
    .groupby("uid")["item_id"]
    .apply(lambda x: set(map(int, x)))
    .to_dict()
)

len(test_user_items)

9778

In [14]:
def dcg_at_k(recs, relevant):
    score = 0.0

    for i, item_id in enumerate(recs, start=1):
        if item_id in relevant:
            score += 1 / np.log2(i + 1)

    return score


def metrics_at_k(recommender, k):
    hits = []
    recalls = []
    ndcgs = []

    recs_global = recommender(k=k)

    for uid, relevant in test_user_items.items():
        seen = train_user_seen.get(uid, set())
        recs = [x for x in recs_global if x not in seen][:k]

        hit_count = len(set(recs) & relevant)

        hits.append(1 if hit_count > 0 else 0)
        recalls.append(hit_count / len(relevant))

        ideal_len = min(len(relevant), k)
        idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_len + 1))
        ndcgs.append(dcg_at_k(recs, relevant) / idcg if idcg > 0 else 0)

    return {
        "hit_rate": float(np.mean(hits)),
        "recall": float(np.mean(recalls)),
        "ndcg": float(np.mean(ndcgs)),
    }


def list_stats(recs):
    clusters_ = [item_to_cluster[x] for x in recs]
    pop = [item_popularity.get(x, 0.0) for x in recs]

    return {
        "unique_items": len(set(recs)),
        "unique_clusters": len(set(clusters_)),
        "cluster_coverage": len(set(clusters_)) / NUM_CLUSTERS,
        "mean_popularity": float(np.mean(pop)),
        "min_popularity": float(np.min(pop)),
        "max_popularity": float(np.max(pop)),
    }

In [15]:
rows = []

for k in K_LIST:
    for name, recommender in [
        ("TopPopular", recommend_top_popular),
        ("ClusterTopPopular", recommend_cluster_div),
    ]:
        recs = recommender(k=k)

        rows.append({
            "model": name,
            "k": k,
            **metrics_at_k(recommender, k),
            **list_stats(recs),
        })

metrics = pd.DataFrame(rows)
metrics

,model,k,hit_rate,recall,ndcg,unique_items,unique_clusters,cluster_coverage,mean_popularity,min_popularity,max_popularity
0,TopPopular,10,0.091021,0.000453,0.010342,10,7,0.233333,18.635818,18.291201,19.658752
1,ClusterTopPopular,10,0.108611,0.000620,0.012804,10,10,0.333333,18.424905,17.892334,19.658752
2,TopPopular,20,0.219370,0.001598,0.013094,20,13,0.433333,18.382088,18.048623,19.658752
3,ClusterTopPopular,20,0.170894,0.001053,0.011642,20,20,0.666667,18.014738,16.735004,19.658752
4,TopPopular,50,0.378810,0.004179,0.014116,50,17,0.566667,17.930507,17.353028,19.658752
5,ClusterTopPopular,50,0.336674,0.003073,0.012459,50,30,1.000000,17.185646,13.159535,19.658752
6,TopPopular,100,0.502250,0.008045,0.013966,100,18,0.600000,17.540519,16.960780,19.658752
7,ClusterTopPopular,100,0.497955,0.006822,0.013682,100,30,1.000000,16.612733,11.305286,19.658752
8,TopPopular,500,0.751994,0.027309,0.019895,500,23,0.766667,16.466056,15.725976,19.658752
9,ClusterTopPopular,500,0.713847,0.019342,0.015289,500,30,1.000000,15.204414,9.124782,19.658752


In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    "item_to_cluster.pkl": item_to_cluster,
    "item_popularity.pkl": item_popularity,
    "cluster_top_items.pkl": cluster_top_items,
    "ordered_clusters.pkl": ordered_clusters,
    "cluster_score.pkl": cluster_score,
    "kmeans.pkl": kmeans,
}

for name, obj in artifacts.items():
    with open(OUTPUT_DIR / name, "wb") as f:
        pickle.dump(obj, f)

config = {
    "model": "ClusterTopPopular",
    "events_data_dir": "flat/50m",
    "events_file": "multi_event.parquet",
    "embeddings_file": "embeddings.parquet",
    "embedding_column": EMB_COL,
    "num_clusters": NUM_CLUSTERS,
    "top_per_cluster": TOP_PER_CLUSTER,
    "random_state": RANDOM_STATE,
    "listen_min": LISTEN_MIN,
    "test_frac_by_time": TEST_FRAC_BY_TIME,
    "popularity": "log1p(listen) + 3*log1p(active_like) - 3*log1p(active_dislike)",
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

metrics.to_csv(OUTPUT_DIR / "offline_metrics.csv", index=False)

list(OUTPUT_DIR.iterdir())

[WindowsPath('model-cold-start/cluster_score.pkl'),
 WindowsPath('model-cold-start/cluster_top_items.pkl'),
 WindowsPath('model-cold-start/config.json'),
 WindowsPath('model-cold-start/item_popularity.pkl'),
 WindowsPath('model-cold-start/item_to_cluster.pkl'),
 WindowsPath('model-cold-start/kmeans.pkl'),
 WindowsPath('model-cold-start/offline_metrics.csv'),
 WindowsPath('model-cold-start/ordered_clusters.pkl')]